# Algoritmo de Colonia de Hormigas

## Descripción General

El **Problema del Viajante de Comercio** es uno de los problemas de optimización combinatoria más famosos y estudiados en Ciencias de la Computación e Investigación Operativa.

**Ejemplo visual con 5 ciudades (A, B, C, D, E):**

```
        A
       / \
      E   B
      |   |
      D - C
```

La hormiga parte de A y debe visitar B, C, D y E antes de regresar a A, eligiendo el camino de menor coste total.

### Objetivos

| Objetivo | Descripción |
|----------|-------------|
| **Minimizar distancia** | Encontrar el recorrido de longitud total mínima |
| **Solución completa** | El recorrido debe incluir **todas** las ciudades |
| **Retorno al origen** | El ciclo debe cerrarse en la ciudad inicial |
| **Escalabilidad** | Aplicable a $n$ ciudades de forma general |


### Limitaciones y Complejidad

El TSP pertenece a la clase de problemas **NP-difíciles**, lo que implica:

- El número de recorridos posibles para $n$ ciudades es **(n-1)! / 2**
- Una búsqueda exhaustiva es **computacionalmente impracticable** para $n$ grande

> *"Para 100 ciudades el número de recorridos posibles es 100!, o sea más de $10^{157}$ recorridos posibles, una exploración exhaustiva de todos ellos es impracticable."*  

Por ello se recurre a **heurísticas y metaheurísticas** que ofrecen soluciones aproximadas en tiempo razonable.


### Trabajos Previos y Contexto Histórico

| Año | Aportación |
|-----|------------|
| **1930s** | Formulación formal del TSP |
| **1954** | Algoritmo exacto de Dantzig, Fulkerson y Johnson|
| **1972** | Demostración de NP-completitud |
| **1991** | Algoritmo de Colonia de Hormigas — Marco Dorigo |
| **1995** | *Particle Swarm Optimization* — Kennedy & Eberhart |
| **2000s** | Variantes como ACS, MMAS, mejoras híbridas con algoritmos genéticos |

 comportamiento de las hormigas reales para encontrar caminos cortos mediante el uso de **feromonas**.


## 2. Fundamento del Algoritmo de Colonia de Hormigas (ACO)

### ¿Cómo funciona?

1. Cada **hormiga artificial** construye un recorrido completo eligiendo ciudades de forma probabilística.
2. La probabilidad de elegir una ciudad depende de:
   - **Feromonas** ($\alpha$): rastros dejados por hormigas anteriores en caminos buenos.
   - **Visibilidad** ($\beta$): inverso de la distancia (ciudades más cercanas son más atractivas).
3. Al finalizar su recorrido, las hormigas que encontraron **mejores caminos depositan más feromonas**.
4. Las feromonas se **evaporan** con el tiempo (coeficiente $\gamma$), evitando convergencia prematura.
5. Iteración tras iteración, las hormigas convergen hacia el **camino óptimo o casi óptimo**.

### Fórmula de probabilidad de transición

$$P(i \to j) = \frac{(1 + \tau_{ij})^\alpha \cdot (1/d_{ij})^\beta}{\sum_{k \notin visitadas} (1 + \tau_{ik})^\alpha \cdot (1/d_{ik})^\beta}$$

## 3. Código: Algoritmo de Colonia de Hormigas

In [ ]:
import random
import sys
import math

def matrizDistancias(nCiud, distanciaMaxima):
    matriz = [[0 for i in range(nCiud)] for j in range(nCiud)]

    for i in range(nCiud):
        for j in range(i):
            matriz[i][j] = distanciaMaxima * random.random()
            matriz[j][i] = matriz[i][j]

    return matriz

def eligeCiudad(dists, ferom, visitadas):

    listaPesos  = []
    disponibles = []
    actual = visitadas[-1]

    alfa = 1.0
    beta = 0.5

    for i in range(len(dists)):
        if i not in visitadas:
            fer  = math.pow((1.0 + ferom[actual][i]), alfa)
            peso = math.pow(1.0 / dists[actual][i], beta) * fer
            disponibles.append(i)
            listaPesos.append(peso)

    valor      = random.random() * sum(listaPesos)
    acumulado  = 0.0
    i          = -1
    while valor > acumulado:
        i         += 1
        acumulado += listaPesos[i]

    return disponibles[i]

def eligeCamino(distancias, feromonas):
    # La ciudad inicial siempre es la 0
    camino     = [0]
    longCamino = 0

    while len(camino) < len(distancias):
        ciudad      = eligeCiudad(distancias, feromonas, camino)
        longCamino += distancias[camino[-1]][ciudad]
        camino.append(ciudad)

    longCamino += distancias[camino[-1]][0]
    camino.append(0)

    return (camino, longCamino)

def rastroFeromonas(feromonas, camino, dosis):
    for i in range(len(camino) - 1):
        feromonas[camino[i]][camino[i + 1]] += dosis

def evaporaFeromonas(feromonas):
    for lista in feromonas:
        for i in range(len(lista)):
            lista[i] *= 0.9


def hormigas(distancias, iteraciones, distMedia):
    # Primero se crea una matriz de feromonas vacía
    n          = len(distancias)
    feromonas  = [[0 for i in range(n)] for j in range(n)]

    mejorCamino     = []
    longMejorCamino = sys.maxsize

    for iter in range(iteraciones):
        (camino, longCamino) = eligeCamino(distancias, feromonas)

        if longCamino <= longMejorCamino:
            mejorCamino     = camino
            longMejorCamino = longCamino

        rastroFeromonas(feromonas, camino, distMedia / longCamino)

        evaporaFeromonas(feromonas)
    return (mejorCamino, longMejorCamino)

## 4. Prueba básica del algoritmo

Configuración del libro: **10 ciudades**, distancia máxima **10**, **1000 iteraciones**.

In [ ]:
random.seed(42) 

numCiudades   = 10
distanciaMaxima = 10
ciudades      = matrizDistancias(numCiudades, distanciaMaxima)

iteraciones  = 1000
distMedia    = numCiudades * distanciaMaxima / 2
(camino, longCamino) = hormigas(ciudades, iteraciones, distMedia)

print("Camino:", camino)
print("Longitud del camino:", round(longCamino, 4))

## 5. Experimento: efecto del número de iteraciones

In [ ]:
import matplotlib.pyplot as plt

random.seed(0)

numCiudades    = 10
distanciaMaxima = 10
ciudades       = matrizDistancias(numCiudades, distanciaMaxima)
distMedia      = numCiudades * distanciaMaxima / 2

puntos_iter    = [10, 50, 100, 200, 300, 400, 500]
longitudes     = []

for iters in puntos_iter:
    resultados = []
    for _ in range(5):   # 5 repeticiones para promediar
        _, long = hormigas(ciudades, iters, distMedia)
        resultados.append(long)
    longitudes.append(sum(resultados) / len(resultados))

plt.figure(figsize=(8, 4))
plt.plot(puntos_iter, longitudes, marker='o', color='steelblue', linewidth=2)
plt.xlabel('Número de iteraciones (hormigas)')
plt.ylabel('Longitud media del camino')
plt.title('Efecto del número de iteraciones\n(Figura 58 — Benítez Iglésias, 2014)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print("La longitud media baja rápidamente y se estabiliza, como describe el libro.")

## 6. Visualización del mejor camino encontrado

In [ ]:
import matplotlib.pyplot as plt
import random

random.seed(42)

n_ciudades = 10
coordenadas = [(random.uniform(0, 10), random.uniform(0, 10)) for _ in range(n_ciudades)]

# Calcular distancias reales entre coordenadas
def dist_euclidiana(c1, c2):
    return math.sqrt((c1[0]-c2[0])**2 + (c1[1]-c2[1])**2)

dist_matrix = [[dist_euclidiana(coordenadas[i], coordenadas[j])
                if i != j else 0.01   # evitar división por cero
                for j in range(n_ciudades)]
               for i in range(n_ciudades)]

distMedia = n_ciudades * 5 / 2
camino, longCamino = hormigas(dist_matrix, 500, distMedia)

fig, ax = plt.subplots(figsize=(7, 6))
xs = [coordenadas[c][0] for c in camino]
ys = [coordenadas[c][1] for c in camino]

ax.plot(xs, ys, 'b-o', linewidth=1.5, markersize=8, label='Ruta ACO')
ax.plot(xs[0], ys[0], 'gs', markersize=14, label='Inicio/Fin (Ciudad 0)')

for i, (x, y) in enumerate(coordenadas):
    ax.annotate(f'  {i}', (x, y), fontsize=11, fontweight='bold')

ax.set_title(f'Mejor ruta encontrada\nLongitud: {longCamino:.3f}', fontsize=13)
ax.set_xlabel('X'); ax.set_ylabel('Y')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Orden de visita: {' → '.join(str(c) for c in camino)}")
print(f"Longitud total: {longCamino:.4f}")